# 📦 Malicious NPM Package Detection — Benchmark Evaluation

**Dataset:** 246 packages (142 malicious / 104 benign)  
**Pipeline:** Structural Analysis → Semantic Analysis → Verification → Final Classification  

---
## Research Questions
- **RQ1:** How does structural analysis of npm packages improve the LLM's ability to identify potentially malicious behaviors?
- **RQ2:** To what extent can LLMs accurately detect and classify behaviors from flagged packages?
- **RQ3:** Can LLMs reconstruct attack chains from detected behaviors, providing a coherent view of the package's threat?

---
## Section 0 — Setup & Dataset Overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support, accuracy_score,
    roc_curve, auc, precision_recall_curve
)
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Plotting style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#2e3347',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linewidth':   0.8,
    'font.family':      'monospace',
    'figure.dpi':       130,
})

MAL_COLOR = '#ff6b6b'   # red  — malicious
BEN_COLOR = '#51cf66'   # green — benign
ACC_COLOR = '#339af0'   # blue  — accuracy / neutral
WARN_COLOR = '#fcc419'  # yellow — warning / partial

print('✅ Libraries loaded.')

In [ ]:
# ── CHANGE THESE PATHS ──────────────────────────────────────────────────────
STRUCTURAL_CSV   = 'structural_analysis_layer_log.csv'
SEMANTIC_CSV     = 'semantic_analysis.csv'
VERIFICATION_CSV = 'verification_analysis.csv'
# ────────────────────────────────────────────────────────────────────────────

df_struct = pd.read_csv(STRUCTURAL_CSV)
df_sem    = pd.read_csv(SEMANTIC_CSV)
df_ver    = pd.read_csv(VERIFICATION_CSV)

# Normalise label column to lowercase
for df in [df_struct, df_sem, df_ver]:
    if 'label' in df.columns:
        df['label'] = df['label'].str.lower().str.strip()

print(f'Structural CSV  : {len(df_struct):>4} rows')
print(f'Semantic CSV    : {len(df_sem):>4} rows')
print(f'Verification CSV: {len(df_ver):>4} rows')

In [ ]:
# ── Dataset overview ────────────────────────────────────────────────────────
label_counts = df_struct['label'].value_counts()
n_mal = label_counts.get('malicious', 0)
n_ben = label_counts.get('benign', 0)
n_total = len(df_struct)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Section 0 — Dataset Overview', fontsize=13, fontweight='bold', color='white', y=1.01)

# Pie chart
ax = axes[0]
wedges, texts, autotexts = ax.pie(
    [n_mal, n_ben],
    labels=['Malicious', 'Benign'],
    colors=[MAL_COLOR, BEN_COLOR],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(edgecolor='#0f1117', linewidth=2)
)
for t in autotexts:
    t.set_color('white'); t.set_fontsize(11)
ax.set_title(f'Label Distribution (n={n_total})', color='white')

# Bar chart
ax2 = axes[1]
bars = ax2.bar(['Malicious', 'Benign'], [n_mal, n_ben],
               color=[MAL_COLOR, BEN_COLOR], edgecolor='#0f1117', linewidth=1.5, width=0.5)
for bar, v in zip(bars, [n_mal, n_ben]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(v), ha='center', va='bottom', fontsize=12, fontweight='bold', color='white')
ax2.set_ylabel('Count')
ax2.set_title('Absolute Package Count', color='white')
ax2.grid(axis='y', alpha=0.4)
ax2.set_ylim(0, max(n_mal, n_ben) * 1.15)

plt.tight_layout()
plt.savefig('fig0_dataset_overview.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()
print(f'Malicious: {n_mal} | Benign: {n_ben} | Total: {n_total}')

---
## Section 1 — RQ1: Structural Analysis Evaluation
> *How does structural analysis improve the LLM's ability to identify potentially malicious behaviors?*

In [ ]:
# ── 1A. Routing Effectiveness (Step 1 as pre-filter) ────────────────────────
df_s1 = df_struct.copy()
df_s1['y_true']  = (df_s1['label'] == 'malicious').astype(int)
df_s1['routing'] = df_s1['routing'].str.lower().str.strip()
df_s1['y_pred']  = (df_s1['routing'] == 'flag').astype(int)

prec, rec, f1, _ = precision_recall_fscore_support(df_s1['y_true'], df_s1['y_pred'], average='binary')
acc = accuracy_score(df_s1['y_true'], df_s1['y_pred'])
cm  = confusion_matrix(df_s1['y_true'], df_s1['y_pred'])

print('=== Step 1 — Routing Performance ===')
print(f'  Accuracy  : {acc:.4f}')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}   ← critical: missed malicious')
print(f'  F1 Score  : {f1:.4f}')
print()
print(classification_report(df_s1['y_true'], df_s1['y_pred'],
                             target_names=['Benign', 'Malicious']))

In [ ]:
# ── 1B. Confusion Matrix + Metrics bar ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle('RQ1 — Step 1: Structural Routing Performance', fontsize=13, fontweight='bold', color='white')

# Confusion matrix
ax = axes[0]
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn',
            xticklabels=['Pred: Pass', 'Pred: Flag'],
            yticklabels=['True: Benign', 'True: Malicious'],
            ax=ax, linewidths=0.5, linecolor='#0f1117',
            annot_kws={'size': 14, 'weight': 'bold'})
ax.set_title('Confusion Matrix — Routing', color='white')
ax.tick_params(colors='white')

# Metrics bar
ax2 = axes[1]
metrics = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}
colors  = [ACC_COLOR, BEN_COLOR, WARN_COLOR, MAL_COLOR]
bars = ax2.bar(metrics.keys(), metrics.values(), color=colors,
               edgecolor='#0f1117', linewidth=1.2, width=0.55)
for bar, val in zip(bars, metrics.values()):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', fontsize=11, fontweight='bold', color='white')
ax2.set_ylim(0, 1.12)
ax2.set_ylabel('Score')
ax2.set_title('Routing Metrics', color='white')
ax2.grid(axis='y', alpha=0.3)
ax2.axhline(0.9, color='white', linestyle='--', linewidth=0.8, alpha=0.4, label='0.9 baseline')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig1a_routing_performance.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
# ── 1C. Risk Score Distribution (mal vs ben) ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle('RQ1 — Structural Signal Quality', fontsize=13, fontweight='bold', color='white')

mal_risk = df_s1[df_s1['label'] == 'malicious']['risk_score'].dropna()
ben_risk = df_s1[df_s1['label'] == 'benign']['risk_score'].dropna()

# Box plot
ax = axes[0]
bp = ax.boxplot([mal_risk, ben_risk], patch_artist=True,
                labels=['Malicious', 'Benign'],
                medianprops=dict(color='white', linewidth=2))
bp['boxes'][0].set_facecolor(MAL_COLOR + '88')
bp['boxes'][1].set_facecolor(BEN_COLOR + '88')
for w in bp['whiskers'] + bp['caps'] + bp['fliers']:
    w.set_color('#8b949e')
ax.set_ylabel('Risk Score')
ax.set_title('Risk Score Distribution', color='white')
ax.grid(axis='y', alpha=0.3)

# KDE / histogram overlay
ax2 = axes[1]
ax2.hist(mal_risk, bins=20, color=MAL_COLOR, alpha=0.6, label='Malicious', density=True)
ax2.hist(ben_risk, bins=20, color=BEN_COLOR, alpha=0.6, label='Benign', density=True)
ax2.set_xlabel('Risk Score')
ax2.set_ylabel('Density')
ax2.set_title('Risk Score Histogram', color='white')
ax2.legend()
ax2.grid(alpha=0.3)

# Mann-Whitney U test
u_stat, p_val = stats.mannwhitneyu(mal_risk, ben_risk, alternative='two-sided')
ax2.text(0.05, 0.92, f'Mann-Whitney U p={p_val:.4f}',
         transform=ax2.transAxes, fontsize=8, color=WARN_COLOR,
         bbox=dict(boxstyle='round', facecolor='#21262d', alpha=0.8))

plt.tight_layout()
plt.savefig('fig1b_risk_score_distribution.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()

print(f'Mal risk score — mean: {mal_risk.mean():.3f}, median: {mal_risk.median():.3f}')
print(f'Ben risk score — mean: {ben_risk.mean():.3f}, median: {ben_risk.median():.3f}')
print(f'Mann-Whitney U statistic: {u_stat:.1f}, p-value: {p_val:.6f}')

In [ ]:
# ── 1D. Correlation: Structural risk_score ↔ Semantic max_confidence ───────
# Merge on package_name + version
df_merge = df_struct[['package_name', 'version', 'label', 'risk_score', 'confidence',
                        'confirmed_count', 'supporting_count', 'routing']].merge(
    df_sem[['package_name', 'version', 'max_confidence', 'total_behaviors', 'parse_success']],
    on=['package_name', 'version'], how='inner'
)

df_valid = df_merge[df_merge['parse_success'] == True].copy()

corr_rs_mc, p_rs_mc = stats.pearsonr(
    df_valid['risk_score'].fillna(0),
    df_valid['max_confidence'].fillna(0)
)
corr_cc_tb, p_cc_tb = stats.pearsonr(
    df_valid['confirmed_count'].fillna(0),
    df_valid['total_behaviors'].fillna(0)
)

print('=== RQ1: Correlation — Structural ↔ Semantic ===')
print(f'  risk_score ↔ LLM max_confidence  : r = {corr_rs_mc:.4f}  (p={p_rs_mc:.4f})')
print(f'  confirmed_count ↔ total_behaviors: r = {corr_cc_tb:.4f}  (p={p_cc_tb:.4f})')
print()
print('Interpretation:')
print(f'  Positive r → structural signals guide LLM confidence effectively')
print(f'  p < 0.05   → statistically significant')

In [ ]:
# ── 1E. Correlation scatter plot ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle('RQ1 — Structural ↔ Semantic Correlation', fontsize=13, fontweight='bold', color='white')

for ax, x_col, y_col, corr, p, title in [
    (axes[0], 'risk_score', 'max_confidence', corr_rs_mc, p_rs_mc, 'Risk Score ↔ LLM Confidence'),
    (axes[1], 'confirmed_count', 'total_behaviors', corr_cc_tb, p_cc_tb, 'Confirmed Signals ↔ Behaviors Detected'),
]:
    colors_pt = df_valid['label'].map({'malicious': MAL_COLOR, 'benign': BEN_COLOR})
    ax.scatter(df_valid[x_col], df_valid[y_col], c=colors_pt, alpha=0.65, s=40, edgecolors='none')

    # Regression line
    m, b = np.polyfit(df_valid[x_col].fillna(0), df_valid[y_col].fillna(0), 1)
    xs = np.linspace(df_valid[x_col].min(), df_valid[x_col].max(), 100)
    ax.plot(xs, m*xs + b, color=ACC_COLOR, linewidth=1.8, linestyle='--', alpha=0.8)

    ax.set_xlabel(x_col.replace('_', ' ').title())
    ax.set_ylabel(y_col.replace('_', ' ').title())
    ax.set_title(title, color='white')
    ax.grid(alpha=0.3)
    sig = '✓ Significant' if p < 0.05 else '✗ Not significant'
    ax.text(0.05, 0.92, f'r = {corr:.3f}  p = {p:.4f}\n{sig}',
            transform=ax.transAxes, fontsize=8.5, color=WARN_COLOR,
            bbox=dict(boxstyle='round', facecolor='#21262d', alpha=0.8))

# Legend
handles = [mpatches.Patch(color=MAL_COLOR, label='Malicious'),
           mpatches.Patch(color=BEN_COLOR, label='Benign')]
fig.legend(handles=handles, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.05))

plt.tight_layout()
plt.savefig('fig1c_correlation.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()

---
## Section 2 — RQ2: Semantic Analysis Evaluation
> *To what extent can LLMs accurately detect and classify behaviors from flagged packages?*

In [ ]:
# ── 2A. Parse Success Rate ───────────────────────────────────────────────────
parse_rate = df_sem['parse_success'].value_counts(normalize=True)
parse_abs  = df_sem['parse_success'].value_counts()

print('=== Step 2 — Parse Success Rate ===')
print(parse_abs.to_string())
print(f'\nSuccess rate: {parse_rate.get(True, 0):.2%}')

# Parse success by label
print('\nParse success by label:')
print(df_sem.groupby('label')['parse_success'].value_counts(normalize=True).unstack().fillna(0).to_string())

In [ ]:
# ── 2B. Threshold sweep — F1 / Precision / Recall vs confidence threshold ───
df_s2 = df_sem.copy()
df_s2['y_true'] = (df_s2['label'] == 'malicious').astype(int)
df_s2['max_confidence'] = df_s2['max_confidence'].fillna(0)
# packages with total_behaviors == 0 → predict benign regardless of confidence
df_s2.loc[df_s2.get('total_behaviors', pd.Series([1]*len(df_s2))) == 0, 'max_confidence'] = 0

thresholds = np.arange(0.0, 1.01, 0.05)
records = []
for t in thresholds:
    y_pred = (df_s2['max_confidence'] >= t).astype(int)
    if y_pred.sum() == 0:
        continue
    p, r, f, _ = precision_recall_fscore_support(df_s2['y_true'], y_pred, average='binary', zero_division=0)
    records.append({'threshold': t, 'precision': p, 'recall': r, 'f1': f})

df_thresh = pd.DataFrame(records)
best_row  = df_thresh.loc[df_thresh['f1'].idxmax()]

print(f'Best threshold: {best_row["threshold"]:.2f}  →  F1={best_row["f1"]:.4f}  '
      f'Precision={best_row["precision"]:.4f}  Recall={best_row["recall"]:.4f}')

In [ ]:
# ── 2C. Threshold plot + Confidence distribution ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('RQ2 — Semantic Analysis (LLM) Performance', fontsize=13, fontweight='bold', color='white')

# Threshold sweep
ax = axes[0]
ax.plot(df_thresh['threshold'], df_thresh['f1'],        color=MAL_COLOR,  lw=2, label='F1')
ax.plot(df_thresh['threshold'], df_thresh['precision'], color=BEN_COLOR,  lw=2, label='Precision')
ax.plot(df_thresh['threshold'], df_thresh['recall'],    color=WARN_COLOR, lw=2, label='Recall')
ax.axvline(best_row['threshold'], color='white', linestyle='--', linewidth=1.2, alpha=0.6,
           label=f'Best t={best_row["threshold"]:.2f}')
ax.set_xlabel('Confidence Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold Sweep', color='white')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)

# Confidence distribution by label
ax2 = axes[1]
for label, color in [('malicious', MAL_COLOR), ('benign', BEN_COLOR)]:
    vals = df_s2[df_s2['label'] == label]['max_confidence']
    ax2.hist(vals, bins=20, color=color, alpha=0.65, label=label.capitalize(), density=True)
ax2.axvline(best_row['threshold'], color='white', linestyle='--', linewidth=1.2, alpha=0.7,
            label=f'Best threshold')
ax2.set_xlabel('Max Confidence')
ax2.set_ylabel('Density')
ax2.set_title('LLM Confidence Distribution', color='white')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fig2a_semantic_performance.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
# ── 2D. Step 2 final metrics at best threshold ───────────────────────────────
y_pred_best = (df_s2['max_confidence'] >= best_row['threshold']).astype(int)
cm2 = confusion_matrix(df_s2['y_true'], y_pred_best)

print('=== Step 2 — Semantic Analysis @ Best Threshold ===')
print(classification_report(df_s2['y_true'], y_pred_best,
                             target_names=['Benign', 'Malicious']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle('RQ2 — Semantic Analysis Final Metrics', fontsize=13, fontweight='bold', color='white')

# Confusion matrix
sns.heatmap(cm2, annot=True, fmt='d', cmap='RdYlGn',
            xticklabels=['Pred: Benign', 'Pred: Malicious'],
            yticklabels=['True: Benign', 'True: Malicious'],
            ax=axes[0], linewidths=0.5, linecolor='#0f1117',
            annot_kws={'size': 14, 'weight': 'bold'})
axes[0].set_title(f'Confusion Matrix (t={best_row["threshold"]:.2f})', color='white')
axes[0].tick_params(colors='white')

# Behavior category distribution
ax2 = axes[1]
all_cats = df_sem['behavior_categories'].dropna().str.split(';').explode().str.strip()
cat_counts = all_cats.value_counts().head(10)
bars = ax2.barh(cat_counts.index[::-1], cat_counts.values[::-1],
                color=MAL_COLOR, alpha=0.8, edgecolor='#0f1117')
ax2.set_xlabel('Count')
ax2.set_title('Top Behavior Categories Detected', color='white')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('fig2b_semantic_confusion_categories.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()

---
## Section 3 — RQ3: Verification & Attack Chain Evaluation
> *Can LLMs reconstruct attack chains from detected behaviors, providing a coherent view of the package's threat?*

In [ ]:
# ── 3A. Verification coverage ────────────────────────────────────────────────
# Not all packages reach Step 3 (those with total_behaviors == 0 are skipped)
n_reached_step3 = len(df_ver)
n_total_step2   = len(df_sem)
print(f'Packages reaching Step 3: {n_reached_step3} / {n_total_step2} ({n_reached_step3/n_total_step2:.1%})')

# Step 3 label distribution
if 'label' in df_ver.columns:
    print('\nStep 3 label distribution:')
    print(df_ver['label'].value_counts().to_string())
else:
    # Try to recover label from structural CSV
    df_ver = df_ver.merge(
        df_struct[['package_name', 'version', 'label']],
        on=['package_name', 'version'], how='left'
    )
    print('Label merged from structural CSV.')

In [ ]:
# ── 3B. Verdict accuracy ─────────────────────────────────────────────────────
df_s3 = df_ver.copy()
df_s3['label']   = df_s3['label'].str.lower().str.strip()
df_s3['verdict'] = df_s3['verdict'].str.upper().str.strip()
df_s3['y_true']  = (df_s3['label'] == 'malicious').astype(int)
df_s3['y_pred']  = (df_s3['verdict'] == 'MALICIOUS').astype(int)

prec3, rec3, f1_3, _ = precision_recall_fscore_support(df_s3['y_true'], df_s3['y_pred'], average='binary', zero_division=0)
acc3 = accuracy_score(df_s3['y_true'], df_s3['y_pred'])
cm3  = confusion_matrix(df_s3['y_true'], df_s3['y_pred'])

print('=== Step 3 — Verdict Performance ===')
print(f'  Accuracy  : {acc3:.4f}')
print(f'  Precision : {prec3:.4f}')
print(f'  Recall    : {rec3:.4f}')
print(f'  F1 Score  : {f1_3:.4f}')
print()
print(classification_report(df_s3['y_true'], df_s3['y_pred'],
                             target_names=['Benign', 'Malicious']))

In [ ]:
# ── 3C. Chain coherence + chain score ────────────────────────────────────────
df_s3['is_coherent_chain'] = df_s3['is_coherent_chain'].astype(str).str.strip()
# Normalise True/False strings
df_s3['coherent'] = df_s3['is_coherent_chain'].map(
    {'True': True, 'False': False, 'true': True, 'false': False, True: True, False: False}
)

coherence_by_label = df_s3.groupby('label')['coherent'].value_counts(normalize=True).unstack().fillna(0)
print('=== Chain Coherence Rate by Label ===')
print(coherence_by_label.to_string())

overall_coherence = df_s3['coherent'].mean()
print(f'\nOverall coherence rate: {overall_coherence:.2%}')

print('\nChain Score by Label:')
print(df_s3.groupby('label')['chain_score'].describe().to_string())

In [ ]:
# ── 3D. Visualization — Verdict CM + Chain score + Coherence ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle('RQ3 — Verification & Attack Chain Evaluation', fontsize=13, fontweight='bold', color='white')

# Confusion matrix
sns.heatmap(cm3, annot=True, fmt='d', cmap='RdYlGn',
            xticklabels=['Pred: Benign', 'Pred: Malicious'],
            yticklabels=['True: Benign', 'True: Malicious'],
            ax=axes[0], linewidths=0.5, linecolor='#0f1117',
            annot_kws={'size': 14, 'weight': 'bold'})
axes[0].set_title('Verdict Confusion Matrix', color='white')
axes[0].tick_params(colors='white')

# Chain score distribution
ax2 = axes[1]
for label, color in [('malicious', MAL_COLOR), ('benign', BEN_COLOR)]:
    vals = df_s3[df_s3['label'] == label]['chain_score'].dropna()
    ax2.hist(vals, bins=15, color=color, alpha=0.65, label=label.capitalize(), density=True)
ax2.set_xlabel('Chain Score')
ax2.set_ylabel('Density')
ax2.set_title('Attack Chain Score Distribution', color='white')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# Coherence rate grouped bar
ax3 = axes[2]
labels_grp = coherence_by_label.index.tolist()
x = np.arange(len(labels_grp))
w = 0.35
coherent_vals   = coherence_by_label.get(True,  pd.Series([0]*len(labels_grp))).values
incoherent_vals = coherence_by_label.get(False, pd.Series([0]*len(labels_grp))).values
ax3.bar(x - w/2, coherent_vals,   w, label='Coherent',   color=BEN_COLOR,  alpha=0.85, edgecolor='#0f1117')
ax3.bar(x + w/2, incoherent_vals, w, label='Incoherent', color=MAL_COLOR,  alpha=0.85, edgecolor='#0f1117')
ax3.set_xticks(x)
ax3.set_xticklabels([l.capitalize() for l in labels_grp])
ax3.set_ylabel('Proportion')
ax3.set_title('Chain Coherence Rate', color='white')
ax3.legend(fontsize=8)
ax3.grid(axis='y', alpha=0.3)
ax3.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig('fig3_verification_evaluation.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()

---
## Section 4 — End-to-End Pipeline Summary

In [ ]:
# ── 4A. Pipeline comparison table ─────────────────────────────────────────────
# Align all three steps on common packages (inner join)
df_all = df_struct[['package_name','version','label','risk_score','routing']].merge(
    df_sem[['package_name','version','max_confidence','total_behaviors','parse_success']],
    on=['package_name','version'], how='left'
).merge(
    df_ver[['package_name','version','verdict','chain_score','is_coherent_chain','confidence']],
    on=['package_name','version'], how='left'
)

df_all['y_true']  = (df_all['label'] == 'malicious').astype(int)
df_all['pred_s1'] = (df_all['routing'].str.lower() == 'flag').astype(int)
df_all['pred_s2'] = (df_all['max_confidence'].fillna(0) >= best_row['threshold']).astype(int)
df_all['pred_s3'] = (df_all['verdict'].fillna('').str.upper() == 'MALICIOUS').astype(int)

summary_rows = []
for step, pred_col in [('Step 1 (Structural)', 'pred_s1'),
                        ('Step 2 (Semantic LLM)', 'pred_s2'),
                        ('Step 3 (Verification)', 'pred_s3')]:
    mask = df_all[pred_col].notna()
    yt = df_all.loc[mask, 'y_true']
    yp = df_all.loc[mask, pred_col]
    if len(yt) == 0: continue
    p, r, f, _ = precision_recall_fscore_support(yt, yp, average='binary', zero_division=0)
    a = accuracy_score(yt, yp)
    tn, fp, fn, tp = confusion_matrix(yt, yp).ravel() if len(np.unique(yp)) > 1 else (0,0,0,0)
    summary_rows.append({'Step': step, 'n': mask.sum(),
                          'Accuracy': a, 'Precision': p, 'Recall': r, 'F1': f,
                          'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn})

df_summary = pd.DataFrame(summary_rows).set_index('Step')
print('=== End-to-End Pipeline Metrics ===')
print(df_summary[['n','Accuracy','Precision','Recall','F1','TP','FP','FN','TN']].to_string())

In [ ]:
# ── 4B. Summary bar chart ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('End-to-End Pipeline — Step-by-Step Metrics Comparison',
             fontsize=13, fontweight='bold', color='white')

steps  = df_summary.index.tolist()
x      = np.arange(len(steps))
width  = 0.2
metric_colors = {'Accuracy': ACC_COLOR, 'Precision': BEN_COLOR, 'Recall': WARN_COLOR, 'F1': MAL_COLOR}

for i, (metric, color) in enumerate(metric_colors.items()):
    vals = df_summary[metric].values
    bars = ax.bar(x + (i - 1.5)*width, vals, width,
                  label=metric, color=color, alpha=0.85, edgecolor='#0f1117')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.2f}', ha='center', fontsize=7.5, color='white')

ax.set_xticks(x)
ax.set_xticklabels(steps, fontsize=10)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.12)
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.axhline(0.9, color='white', linestyle=':', linewidth=0.8, alpha=0.4)

plt.tight_layout()
plt.savefig('fig4_pipeline_summary.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()

---
## Section 5 — Error Analysis (FP / FN Cases)

In [ ]:
# ── 5A. False Positive / False Negative breakdown ────────────────────────────
# Using Step 3 verdict as final classifier (most downstream LLM step)
df_err = df_all[df_all['verdict'].notna()].copy()
df_err['verdict_norm'] = df_err['verdict'].str.upper().str.strip()
df_err['fp'] = (df_err['y_true'] == 0) & (df_err['verdict_norm'] == 'MALICIOUS')
df_err['fn'] = (df_err['y_true'] == 1) & (df_err['verdict_norm'] == 'BENIGN')
df_err['tp'] = (df_err['y_true'] == 1) & (df_err['verdict_norm'] == 'MALICIOUS')
df_err['tn'] = (df_err['y_true'] == 0) & (df_err['verdict_norm'] == 'BENIGN')

fp_cases = df_err[df_err['fp']][['package_name','version','risk_score','max_confidence','chain_score']]
fn_cases = df_err[df_err['fn']][['package_name','version','risk_score','max_confidence','chain_score']]

print(f'False Positives (Benign → predicted Malicious): {len(fp_cases)}')
print(fp_cases.to_string(index=False))
print()
print(f'False Negatives (Malicious → predicted Benign): {len(fn_cases)}')
print(fn_cases.to_string(index=False))

In [ ]:
# ── 5B. Error pattern visualisation ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle('Section 5 — Error Analysis', fontsize=13, fontweight='bold', color='white')

# FP risk score vs FN risk score
ax = axes[0]
cats   = ['TP', 'TN', 'FP', 'FN']
counts = [df_err['tp'].sum(), df_err['tn'].sum(), df_err['fp'].sum(), df_err['fn'].sum()]
colors_err = [BEN_COLOR, ACC_COLOR, WARN_COLOR, MAL_COLOR]
bars = ax.bar(cats, counts, color=colors_err, edgecolor='#0f1117', linewidth=1.2, width=0.5)
for bar, v in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(v), ha='center', fontsize=12, fontweight='bold', color='white')
ax.set_ylabel('Count')
ax.set_title('Prediction Outcome Distribution (Step 3)', color='white')
ax.grid(axis='y', alpha=0.3)

# Risk score of FP vs FN
ax2 = axes[1]
if len(fp_cases) > 0 and len(fn_cases) > 0:
    ax2.scatter(fp_cases['risk_score'], fp_cases['max_confidence'],
                color=WARN_COLOR, s=80, label=f'FP (n={len(fp_cases)})', zorder=3, edgecolors='white', linewidths=0.5)
    ax2.scatter(fn_cases['risk_score'], fn_cases['max_confidence'],
                color=MAL_COLOR,  s=80, label=f'FN (n={len(fn_cases)})', zorder=3, marker='X', edgecolors='white', linewidths=0.5)
    ax2.set_xlabel('Structural Risk Score')
    ax2.set_ylabel('LLM Max Confidence')
    ax2.set_title('FP vs FN — Risk Score vs LLM Confidence', color='white')
    ax2.legend(fontsize=9)
    ax2.grid(alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'Insufficient FP/FN data', ha='center', va='center',
             transform=ax2.transAxes, fontsize=12, color='#8b949e')
    ax2.set_title('FP vs FN Analysis', color='white')

plt.tight_layout()
plt.savefig('fig5_error_analysis.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
# ── Final Summary Print ───────────────────────────────────────────────────────
print('=' * 60)
print('  BENCHMARK SUMMARY')
print('=' * 60)
print(f'  Dataset : {n_total} packages ({n_mal} mal / {n_ben} ben)')
print()
print('  RQ1 — Structural Analysis (Step 1 Routing)')
p1, r1, f1_1, _ = precision_recall_fscore_support(df_s1['y_true'], df_s1['y_pred'], average='binary', zero_division=0)
a1 = accuracy_score(df_s1['y_true'], df_s1['y_pred'])
print(f'    Accuracy={a1:.3f}  Precision={p1:.3f}  Recall={r1:.3f}  F1={f1_1:.3f}')
print(f'    risk_score ↔ LLM confidence  r={corr_rs_mc:.3f} (p={p_rs_mc:.4f})')
print()
print(f'  RQ2 — Semantic Analysis (Step 2, t={best_row["threshold"]:.2f})')
p2, r2, f1_2, _ = precision_recall_fscore_support(df_s2['y_true'], y_pred_best, average='binary', zero_division=0)
a2 = accuracy_score(df_s2['y_true'], y_pred_best)
print(f'    Accuracy={a2:.3f}  Precision={p2:.3f}  Recall={r2:.3f}  F1={f1_2:.3f}')
print(f'    Parse success rate: {parse_rate.get(True, 0):.2%}')
print()
print(f'  RQ3 — Verification / Attack Chain (Step 3)')
print(f'    Accuracy={acc3:.3f}  Precision={prec3:.3f}  Recall={rec3:.3f}  F1={f1_3:.3f}')
print(f'    Chain coherence (overall): {overall_coherence:.2%}')
print()
print(f'  FP={df_err["fp"].sum()}  FN={df_err["fn"].sum()}  TP={df_err["tp"].sum()}  TN={df_err["tn"].sum()}')
print('=' * 60)